In [1584]:
import cv2
import numpy as np
import os
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple, Optional, List, Dict
import random
from PIL import Image, ImageDraw, ImageFilter, ImageFont

# ==================== PHASE 1: YOLO CLASS DEFINITIONS ====================

YOLO_CLASSES = {
    0: "Player Champion",      # Green HP
    1: "Ally Champion",        # Blue HP
    2: "Enemy Champion",       # Red HP
    3: "Turret",               # Structure (no mana)
    4: "Epic Monster",         # Dragon/Baron
    5: "Jungle Monster"        # Krugs, wolves, etc.
}

CLASS_TO_ID = {v: k for k, v in YOLO_CLASSES.items()}

# Directory structure
BASE_DIR = Path("/Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar")
COLORS_DIR = BASE_DIR / "Classified" / "Colors"
FRAMES_DIR = BASE_DIR / "Classified" / "Frames"
BACKGROUNDS_DIR = BASE_DIR / "crops"  # Your in-game backgrounds
OUTPUT_DIR = BASE_DIR / "synthetic_dataset"
IMAGES_DIR = OUTPUT_DIR / "images"
LABELS_DIR = OUTPUT_DIR / "labels"

# Create output directories
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ YOLO Classes defined: {len(YOLO_CLASSES)} types")
print(f"✓ Output will go to: {OUTPUT_DIR}")
print(f"✓ Color assets: {COLORS_DIR}")
print(f"✓ Frame assets: {FRAMES_DIR}")
print(f"✓ Background crops: {BACKGROUNDS_DIR}")

✓ YOLO Classes defined: 6 types
✓ Output will go to: /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/synthetic_dataset
✓ Color assets: /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/Classified/Colors
✓ Frame assets: /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/Classified/Frames
✓ Background crops: /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/crops


In [1585]:
# ==================== PHASE 2: FRAME CONFIGURATION ====================

FRAME_CONFIGS = {
    "standard_champion": {
        "file": "Classified/Frames/healthbar.png",
        "hp_x": 26, "hp_y": 5, "hp_w": 107, "hp_h": 13,
        "mana_y": 19, "mana_h": 5,
        "has_mana": True,
        "yolo_class": "Player Champion",
        "banner_shape": "square", "banner_x": 4, "banner_y": 4,
        "level_x": 5, "level_y": 4, "font_size": 16
    },
    "execute_threshold": {
        "file": "Classified/Frames/execute_treshold_health_frame.png",
        "hp_x": 30, "hp_y": 8, "hp_w": 106, "hp_h": 12,
        "mana_y": 22, "mana_h": 5,
        "has_mana": True,
        "yolo_class": "Player Champion",
        "banner_shape": "rombhus", "banner_x": 8, "banner_y": 5,
        "level_x": 8, "level_y": 6, "font_size": 16
    },
    "invulnerable": {
        "file": "Classified/Frames/frame_invulnerable.png",
        "hp_x": 27, "hp_y": 5, "hp_w": 106, "hp_h": 13,
        "mana_y": 19, "mana_h": 5,
        "has_mana": True,
        "yolo_class": "Player Champion",
        "banner_shape": "shield", "banner_x": 4, "banner_y": 4,
        "level_x": 5, "level_y": 3, "font_size": 16
    },
    "spellshield": {
        "file": "Classified/Frames/frame_spellshield.png",
        "hp_x": 27, "hp_y": 6, "hp_w": 106, "hp_h": 12,
        "mana_y": 20, "mana_h": 5,
        "has_mana": True,
        "yolo_class": "Player Champion",
        "banner_shape": "rombhus", "banner_x": 4, "banner_y": 3,
        "level_x": 5, "level_y": 4, "font_size": 16
    },
    "yuumi_special": {
        "file": "Classified/Frames/healthbar.png", 
        "hp_x": 26, "hp_y": 5, "hp_w": 104, "hp_h": 13,
        "mana_y": 19, "mana_h": 5,
        "has_mana": True,
        "yolo_class": "Player Champion",
        "overlay_file": "Classified/Frames/yummi_health_frame_cropped.png",
        "overlay_crop_w": 38, 
        "overlay_x": -5,
        "overlay_y": -5,
        "banner_shape": "square", "banner_x": 4, "banner_y": 4,
        "level_x": 5, "level_y": 4, "font_size": 16
    },
    "turret": {
        "file": "Classified/Frames/turret_frame.png",
        "hp_x": 11, "hp_y": 6, "hp_w": 168, "hp_h": 15,
        "mana_y": 0, "mana_h": 0,
        "has_mana": False,
        "yolo_class": "Turret"
    },
    "epic_monster": {
        "file": "Classified/Frames/epic_monster.png",
        "hp_x": 12, "hp_y": 20, "hp_w": 168, "hp_h": 15,
        "mana_y": 0, "mana_h": 0,
        "has_mana": False,
        "yolo_class": "Epic Monster",
        "monster_font_size": 16,
        "monster_number_pivot_x": -2,
        "has_monster_number": True,        # NEW: Toggle number rendering
        "monster_number_offset_y": -23      # NEW: Adjust vertical placement
    },
    "jungle_monster_big": {
        "file": "Classified/Frames/jungle_monster_big.png",
        "hp_x": 4, "hp_y": 21, "hp_w": 143, "hp_h": 12,
        "mana_y": 0, "mana_h": 0,
        "has_mana": False,
        "yolo_class": "Jungle Monster",
        "monster_font_size": 16,
        "monster_number_pivot_x": -2,
        "has_monster_number": True,        # NEW: Toggle number rendering
        "monster_number_offset_y": -23      # NEW: Adjust vertical placement
    },
    "structure": {
        "file": "Classified/Frames/structure_frame.png",
        "hp_x": 5, "hp_y": 4, "hp_w": 142, "hp_h": 10,
        "mana_y": 0, "mana_h": 0,
        "has_mana": False,
        "yolo_class": "Turret"
    }
}

def load_and_verify_configs():
    print("\n=== FRAME CONFIG VERIFICATION ===")
    valid_configs = {}
    for frame_name, config in FRAME_CONFIGS.items():
        frame_path = BASE_DIR / config["file"]
        if frame_path.exists():
            img = cv2.imread(str(frame_path), cv2.IMREAD_UNCHANGED)
            if img is not None:
                h, w = img.shape[:2]
                valid_configs[frame_name] = config
                print(f"✓ {frame_name:20} | {w:3}x{h:3} | HP: ({config['hp_x']}, {config['hp_y']}) {config['hp_w']}x{config['hp_h']}")
            else:
                print(f"✗ {frame_name:20} | FAILED TO LOAD")
        else:
            print(f"✗ {frame_name:20} | NOT FOUND: {frame_path}")
    return valid_configs

VALID_FRAMES = load_and_verify_configs()


=== FRAME CONFIG VERIFICATION ===
✓ standard_champion    | 136x 29 | HP: (26, 5) 107x13
✓ execute_threshold    | 140x 35 | HP: (30, 8) 106x12
✓ invulnerable         | 136x 33 | HP: (27, 5) 106x13
✓ spellshield          | 136x 31 | HP: (27, 6) 106x12
✓ yuumi_special        | 136x 29 | HP: (26, 5) 104x13
✓ turret               | 192x 34 | HP: (11, 6) 168x15
✓ epic_monster         | 192x 48 | HP: (12, 20) 168x15
✓ jungle_monster_big   | 151x 43 | HP: (4, 21) 143x12
✓ structure            | 151x 23 | HP: (5, 4) 142x10


In [1586]:
# ==================== PHASE 3: COLOR & ASSET LOADER ====================

@dataclass
class ColorAsset:
    """Represents a color strip asset"""
    name: str
    image: np.ndarray  # Shape: (H, W, 4) with alpha
    color_type: str    # "hp", "mana", "shield", "grey"

class AssetLibrary:
    """Loads and manages all color and special effect assets"""
    
    def __init__(self, colors_dir: Path):
        self.colors_dir = colors_dir
        self.banners_dir = colors_dir.parent / "Banners"
        self.font_path = colors_dir.parent / "Fonts" / "BeaufortForLoL-OTF" / "BeaufortForLoL-Bold.otf"
        self.hp_colors = {}      # team -> [color_assets]
        self.special_effects = {} # effect_type -> asset
        self.banners = {}        # shape_team -> asset
        self.load_assets()
    
    def load_assets(self):
        """Load all color classifications"""
        if not self.colors_dir.exists():
            print(f"⚠ Colors directory not found: {self.colors_dir}")
            return
        
        hp_files = {
            "player": ["green_character_hpcolor.png"],
            "ally": ["ally_hpcolor.png"],
            "enemy": ["enemy_hp_color.png"],
        }
        
        special_files = {
            "shield": "shield_normal.png",
            "grey_health": "grey_health.png",
            "spellshield": "shield_magic_purple.png",
            "kled": "kled_extra_hpcolor.png",
            "mana": "resource_mana_color.png",
            "energy": "resource_energy_color.png",
            "fury": "resource_fury_orange.png",
            "hp_gradient": "hp_green_gradient.png",
        }

        if self.banners_dir.exists():
            banner_files = {
                "square_blue": "square_blue.png",
                "square_red": "square_red.png",
                "rombhus_blue": "rombhus_ally.png",
                "rombhus_red": "rombhus_red.png",
                "shield_blue": "shield_blue.png",
                "shield_red": "shield_red.png",
            }
            
            for key, filename in banner_files.items():
                path = self.banners_dir / filename
                if path.exists():
                    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
                    if img is not None:
                        self.banners[key] = ColorAsset(name=filename, image=img, color_type="banner")
            print(f"✓ Loaded {len(self.banners)} Banner assets")
        else:
            print(f"⚠ Banners directory not found: {self.banners_dir}")

        for team, files in hp_files.items():
            self.hp_colors[team] = []
            for filename in files:
                path = self.colors_dir / filename
                if path.exists():
                    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
                    if img is not None:
                        asset = ColorAsset(name=filename, image=img, color_type="hp")
                        self.hp_colors[team].append(asset)
        
        for effect_type, filename in special_files.items():
            path = self.colors_dir / filename
            if path.exists():
                img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
                if img is not None:
                    asset = ColorAsset(name=filename, image=img, color_type=effect_type)
                    self.special_effects[effect_type] = asset

        print(f"✓ Loaded {len(self.hp_colors)} HP color teams")
        for team, assets in self.hp_colors.items():
            print(f"  - {team:8}: {len(assets)} color variants")
        print(f"✓ Loaded {len(self.special_effects)} special effect assets")
    
    def get_hp_color(self, team: str) -> Optional[ColorAsset]:
        """Get random HP color for a team"""
        if team in self.hp_colors and self.hp_colors[team]:
            return random.choice(self.hp_colors[team])
        return None
    
    def get_special_effect(self, effect_type: str) -> Optional[ColorAsset]:
        """Get special effect asset"""
        return self.special_effects.get(effect_type)
    
    def get_banner(self, shape: str, team: str) -> Optional[ColorAsset]:
        """Gets the correct banner based on shape and team color logic"""
        color = "red" if team == "enemy" else "blue"
        key = f"{shape}_{color}"
        return self.banners.get(key)

# Initialize asset library
assets = AssetLibrary(COLORS_DIR)
print(f"\n✓ Asset Library initialized")

✓ Loaded 6 Banner assets
✓ Loaded 3 HP color teams
  - player  : 1 color variants
  - ally    : 1 color variants
  - enemy   : 1 color variants
✓ Loaded 7 special effect assets

✓ Asset Library initialized


In [1587]:
# ==================== HEALTH BAR FACTORY ====================

class HealthBarFactory:
    """Generates randomized, synthetic health bars by mixing assets"""
    
    def __init__(self, asset_lib: AssetLibrary, frame_configs: Dict):
        self.assets = asset_lib
        self.frames = frame_configs
    
    def _draw_tick_marks(self, hp_strip: np.ndarray, full_hp_w: int, max_hp: int, frame_name: str) -> np.ndarray:
        """Draws tick marks for champions or plating gaps for turrets based on total frame health width."""
        result = hp_strip.copy()
        h = result.shape[0]
        
        # TURRET PLATING LOGIC: Punch transparent gaps
        if frame_name == "turret":
            num_segments = 5
            gap_w = 2
            for i in range(1, num_segments):
                seg_w = full_hp_w / num_segments
                x_pos = int(i * seg_w)
                if x_pos - gap_w >= 0 and x_pos < result.shape[1]:
                    result[:, max(0, x_pos-gap_w):min(x_pos, result.shape[1]), 3] = 0
            return result
        
        if frame_name == "structure":
            return result
        
        if max_hp <= 0: return result
        
        # Epic and Large Jungle Monsters only get full-height lines every 1k health
        if frame_name in ["epic_monster", "jungle_monster_big"]:
            pixels_per_1000 = (full_hp_w / max_hp) * 1000
            current_x = pixels_per_1000
            while current_x < result.shape[1]:
                x_pos = int(current_x)
                cv2.line(result, (x_pos, 0), (x_pos, h), (0, 0, 0, 255), 1)
                current_x += pixels_per_1000
        else:
            # Standard Champion Tick Logic (100 HP ticks + 1000 HP full lines)
            pixels_per_100 = (full_hp_w / max_hp) * 100
            current_x = pixels_per_100
            hp_count = 100
            tick_h = max(1, int(h * 0.4))
            
            while current_x < result.shape[1]:
                x_pos = int(current_x)
                if hp_count % 1000 == 0:
                    cv2.line(result, (x_pos, 0), (x_pos, h), (0, 0, 0, 255), 1)
                else:
                    cv2.line(result, (x_pos, 0), (x_pos, tick_h), (0, 0, 0, 255), 1)
                current_x += pixels_per_100
                hp_count += 100
        return result
    
    def create_healthbar(self, hp_percentage: float, mana_percentage: float, 
                        team: str, frame_name: str,
                        shield_pct: float = 0.0, 
                        spellshield_pct: float = 0.0,
                        grey_hp_pct: float = 0.0,
                        kled_hp_pct: float = 0.0) -> Tuple[np.ndarray, Dict]:
        
        if frame_name not in self.frames:
            return None, None
        
        frame_cfg = self.frames[frame_name]
        frame_path = BASE_DIR / frame_cfg["file"]
        
        frame = cv2.imread(str(frame_path), cv2.IMREAD_UNCHANGED)
        if frame is None: return None, None
        frame = frame.copy()
        
        # NEW: Force target asset colors to look blue if assigned to player or ally teams
        # This covers ally jungle monsters, ally structures, and ally turrets.
        resolved_color_team = team
        if frame_name in ["turret", "structure", "epic_monster", "jungle_monster_big"]:
            if team in ["player", "ally"]:
                resolved_color_team = "ally"  # Ensures blue health bar strip is selected
        
        # Strip shields, grey health, and Kled health entirely for non-champion entities
        is_champion = frame_name not in ["turret", "structure", "epic_monster", "jungle_monster_big"]
        if not is_champion:
            shield_pct = 0.0
            spellshield_pct = 0.0
            grey_hp_pct = 0.0
            kled_hp_pct = 0.0
        
        # Determine Max HP range dynamically
        if frame_name == "epic_monster":
            max_hp = random.randint(2500, 30000)      
        elif frame_name == "jungle_monster_big":
            max_hp = random.randint(500, 3000)        
        else:
            max_hp = random.randint(600, 4000)

        # Derive health percentage and precise health values
        if hp_percentage is not None:
            hp_pct = max(0.0, min(100.0, hp_percentage))
            current_hp = int(round((max_hp * hp_pct) / 100.0))
        else:
            hp_pct = random.uniform(10.0, 100.0)
            current_hp = int(round((max_hp * hp_pct) / 100.0))
        
        # Canvas setup 
        pad = 30
        canvas_h, canvas_w = frame.shape[0] + (pad * 2), frame.shape[1] + (pad * 2)
        result = np.zeros((canvas_h, canvas_w, 4), dtype=np.uint8)
        b_x, b_y = pad, pad 
        
        # 2. Base track background
        empty_bg = np.full((frame_cfg["hp_h"], frame_cfg["hp_w"], 4), (30, 30, 30, 255), dtype=np.uint8)
        empty_roi = result[
            b_y + frame_cfg["hp_y"] : b_y + frame_cfg["hp_y"] + frame_cfg["hp_h"],
            b_x + frame_cfg["hp_x"] : b_x + frame_cfg["hp_x"] + frame_cfg["hp_w"]
        ]
        self._blend_alpha(empty_roi, empty_bg, frame_cfg["hp_w"])
        
        # 3. Draw Active HP Color, Grey Health, and Shields sequentially
        target_hp_w = int(round((frame_cfg["hp_w"] * hp_pct) / 100.0)) if hp_pct < 99.5 else frame_cfg["hp_w"]
        combined_hp_strip = np.zeros((frame_cfg["hp_h"], frame_cfg["hp_w"], 4), dtype=np.uint8)
        
        def overlay_block(strip, width, start_x, asset_name, is_team_color=False):
            if width <= 0 or start_x >= frame_cfg["hp_w"]: 
                return strip, start_x
            
            if is_team_color:
                asset = self.assets.get_hp_color(asset_name)
            else:
                asset = self.assets.get_special_effect(asset_name)
                
            if asset is None: 
                return strip, start_x
            
            w_clamped = min(width, frame_cfg["hp_w"] - start_x)
            if w_clamped <= 0: 
                return strip, start_x
            
            img_to_resize = asset.image.copy()
            
            # Auto-crop dead padding
            gray = cv2.cvtColor(img_to_resize[:, :, :3], cv2.COLOR_BGR2GRAY)
            mask = ((img_to_resize[:, :, 3] > 20) & (gray > 20)).astype(np.uint8) if img_to_resize.shape[2] == 4 else (gray > 20).astype(np.uint8)
            coords = cv2.findNonZero(mask)
            if coords is not None:
                cx, cy, cw, ch = cv2.boundingRect(coords)
                img_to_resize = img_to_resize[:, cx : cx + cw]
            
            # Slice proportional native width to preserve gradients
            native_w = img_to_resize.shape[1]
            crop_w = int(max(1, native_w * (width / frame_cfg["hp_w"])))
            crop_w = min(crop_w, native_w)
            asset_cropped = img_to_resize[:, :crop_w]
            
            layer_resized = cv2.resize(asset_cropped, (w_clamped, frame_cfg["hp_h"]), interpolation=cv2.INTER_LINEAR)
            
            # Alpha blend seamlessly
            if layer_resized.shape[2] == 4:
                alpha = layer_resized[:, :, 3:4] / 255.0
                strip[:, start_x : start_x + w_clamped, :3] = (
                    layer_resized[:, :, :3] * alpha + \
                    strip[:, start_x : start_x + w_clamped, :3] * (1.0 - alpha)
                ).astype(np.uint8)
                strip[:, start_x : start_x + w_clamped, 3] = np.maximum(
                    strip[:, start_x : start_x + w_clamped, 3], layer_resized[:, :, 3]
                )
            else:
                strip[:, start_x : start_x + w_clamped, :3] = layer_resized[:, :, :3]
                strip[:, start_x : start_x + w_clamped, 3] = 255
                
            return strip, start_x + w_clamped

        # A. Base Health (Uses resolved_color_team to handle blue monster/structure overrides)
        current_x = 0
        combined_hp_strip, current_x = overlay_block(combined_hp_strip, target_hp_w, current_x, resolved_color_team, True)
            
        # B. Kled Health
        if kled_hp_pct > 0.0:
            kled_w = int(round((frame_cfg["hp_w"] * kled_hp_pct) / 100.0))
            combined_hp_strip, current_x = overlay_block(combined_hp_strip, kled_w, current_x, "kled", False)
            
        base_hp_end_x = current_x  
            
        # C. Grey Health 
        if grey_hp_pct > 0.0 and base_hp_end_x < frame_cfg["hp_w"]:
            grey_w = int(round((frame_cfg["hp_w"] * grey_hp_pct) / 100.0))
            overlay_block(combined_hp_strip, grey_w, base_hp_end_x, "grey_health", False)
            
        # D. Shields (Append sequentially)
        shield_w = int(round((frame_cfg["hp_w"] * shield_pct) / 100.0))
        combined_hp_strip, current_x = overlay_block(combined_hp_strip, shield_w, base_hp_end_x, "shield", False)
        
        ss_w = int(round((frame_cfg["hp_w"] * spellshield_pct) / 100.0))
        combined_hp_strip, current_x = overlay_block(combined_hp_strip, ss_w, current_x, "spellshield", False)
        
        # E. Render to Canvas
        total_active_fill_w = min(frame_cfg["hp_w"], max(base_hp_end_x, current_x))
        
        if total_active_fill_w > 0:
            filled_strip_cropped = combined_hp_strip[:, :total_active_fill_w]
            hp_with_marks = self._draw_tick_marks(filled_strip_cropped, frame_cfg["hp_w"], max_hp, frame_name)
            
            hp_roi = result[
                b_y + frame_cfg["hp_y"] : b_y + frame_cfg["hp_y"] + frame_cfg["hp_h"],
                b_x + frame_cfg["hp_x"] : b_x + frame_cfg["hp_x"] + total_active_fill_w
            ]
            self._blend_alpha(hp_roi, hp_with_marks, total_active_fill_w)
        
        # FIX: Hole punch the frame over the ENTIRE configured slot width unconditionally.
        # This completely removes that blocky dark-alpha artifact behind low health bars.
        frame[frame_cfg["hp_y"] : frame_cfg["hp_y"] + frame_cfg["hp_h"], \
              frame_cfg["hp_x"] : frame_cfg["hp_x"] + frame_cfg["hp_w"], 3] = 0
        
        # 4. Draw Resource Bar
        if frame_cfg["has_mana"]:
            mana_pct = mana_percentage if mana_percentage is not None else random.uniform(30, 100)
            resource_type = random.choice(["mana", "energy", "fury"])
            resource_asset = self.assets.get_special_effect(resource_type)
            
            if resource_asset:
                mana_cropped = self._crop_color_to_percentage(resource_asset.image, mana_pct)
                actual_mana_w = mana_cropped.shape[1]
                
                mana_roi = result[
                    b_y + frame_cfg["mana_y"] : b_y + frame_cfg["mana_y"] + frame_cfg["mana_h"],
                    b_x + frame_cfg["hp_x"] : b_x + frame_cfg["hp_x"] + actual_mana_w
                ]
                self._blend_alpha(mana_roi, mana_cropped, actual_mana_w)
                frame[frame_cfg["mana_y"] : frame_cfg["mana_y"] + frame_cfg["mana_h"], \
                      frame_cfg["hp_x"] : frame_cfg["hp_x"] + actual_mana_w, 3] = 0
                
        if "banner_shape" in frame_cfg:
            banner_asset = self.assets.get_banner(frame_cfg["banner_shape"], team)
            if banner_asset is not None:
                banner_x = b_x + frame_cfg.get("banner_x", 0)
                banner_y = b_y + frame_cfg.get("banner_y", 0)
                banner_img = banner_asset.image
                
                if banner_y >= 0 and banner_x >= 0:
                    banner_roi = result[
                        banner_y : banner_y + banner_img.shape[0], \
                        banner_x : banner_x + banner_img.shape[1]
                    ]
                    self._blend_alpha(banner_roi, banner_img, banner_img.shape[1])
        
        # 5. Overlay Base Frame
        frame_roi = result[b_y : b_y + frame.shape[0], b_x : b_x + frame.shape[1]]
        self._blend_alpha(frame_roi, frame, frame.shape[1])

        # 5.5 Draw Level Text
        if "level_x" in frame_cfg:
            lvl = str(random.randint(1, 18))
            lvl_x = b_x + frame_cfg["level_x"]
            if len(lvl) == 1: 
                lvl_x += 3  
            lvl_y = b_y + frame_cfg["level_y"]
            
            result = self._draw_level_text(result, lvl, lvl_x, lvl_y, frame_cfg["font_size"])

        # Overhead Monster Sizing & Tracking logic
        if frame_cfg.get("has_monster_number", False):
            m_num = str(current_hp)
            font_sz = frame_cfg.get("monster_font_size", 14)
            
            hp_center_x = b_x + frame_cfg["hp_x"] + (frame_cfg["hp_w"] // 2)
            pivot_center_x = hp_center_x + frame_cfg.get("monster_number_pivot_x", 0)
            approx_text_w = (font_sz // 2) * len(m_num)
            num_x = pivot_center_x - (approx_text_w // 2)
            
            num_y = b_y + frame_cfg["hp_y"] + frame_cfg.get("monster_number_offset_y", -12)
            
            result = self._draw_level_text(result, m_num, num_x, num_y, font_sz)
        
        # 6. Apply Special Overlays
        if "overlay_file" in frame_cfg:
            ov_path = BASE_DIR / frame_cfg["overlay_file"]
            overlay_img = cv2.imread(str(ov_path), cv2.IMREAD_UNCHANGED)
            if overlay_img is not None:
                if "overlay_crop_w" in frame_cfg:
                    overlay_img = overlay_img[:, :frame_cfg["overlay_crop_w"]]
                o_x = b_x + frame_cfg.get("overlay_x", 0)
                o_y = b_y + frame_cfg.get("overlay_y", 0)
                ov_roi = result[o_y : o_y + overlay_img.shape[0], o_x : o_x + overlay_img.shape[1]]
                self._blend_alpha(ov_roi, overlay_img, overlay_img.shape[1])
        
        # 7. Auto-Crop bounds
        coords = cv2.findNonZero(result[:, :, 3]) 
        x, y, w, h = cv2.boundingRect(coords)
        final_result = result[y:y+h, x:x+w]
        
        metadata = {
            "team": team, "frame": frame_name, "hp_pct": hp_pct,
            "hp_x": (b_x + frame_cfg["hp_x"]) - x, "hp_y": (b_y + frame_cfg["hp_y"]) - y,
            "hp_w": frame_cfg["hp_w"], "hp_h": frame_cfg["hp_h"], "width": w, "height": h
        }
        return final_result, metadata

    def _crop_color_to_percentage(self, color_strip: np.ndarray, percentage: float) -> np.ndarray:
        _, crop_w = color_strip.shape[:2]
        target_w = int(crop_w * max(0, min(100, percentage)) / 100)
        return color_strip[:, :max(1, target_w)]
    
    def _blend_alpha(self, dst_roi: np.ndarray, src: np.ndarray, max_width: int):
        if src.shape[0] == 0 or src.shape[1] == 0: return
        h_dst, w_dst = dst_roi.shape[:2]
        src_resized = cv2.resize(src, (min(src.shape[1], w_dst), h_dst), interpolation=cv2.INTER_NEAREST)
        
        if src_resized.shape[2] == 4: 
            alpha = src_resized[:, :, 3:4] / 255.0
            src_rgb = src_resized[:, :, :3]
            for c in range(3):
                dst_roi[:, :src_resized.shape[1], c] = \
                    (src_rgb[:, :, c] * alpha[:, :, 0] + dst_roi[:, :src_resized.shape[1], c] * (1 - alpha[:, :, 0])).astype(np.uint8)
            dst_roi[:, :src_resized.shape[1], 3] = np.maximum(dst_roi[:, :src_resized.shape[1], 3], src_resized[:, :, 3])
        else:
            dst_roi[:, :src_resized.shape[1], :3] = src_resized

    def _draw_level_text(self, result_canvas: np.ndarray, text: str, x: int, y: int, font_size: int) -> np.ndarray:
        """Draws authentic TTF font with a drop shadow, using dynamic font_size"""
        img_rgb = cv2.cvtColor(result_canvas, cv2.COLOR_BGRA2RGBA)
        pil_img = Image.fromarray(img_rgb)
        draw = ImageDraw.Draw(pil_img)
        
        try:
            font = ImageFont.truetype(str(self.assets.font_path), font_size)
        except Exception as e:
            print(f"⚠ Font load failed, using default: {e}")
            font = ImageFont.load_default()
            
        draw.text((x + 1, y + 1), text, font=font, fill=(0, 0, 0, 255))
        draw.text((x, y), text, font=font, fill=(255, 255, 255, 255))
        
        return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGBA2BGRA)


factory = HealthBarFactory(assets, VALID_FRAMES)
print("✓ HealthBar Factory updated successfully.")

✓ HealthBar Factory updated successfully.


In [1588]:
# ==================== DATASET COMPOSITOR ====================
# Simplified version: Save raw health bars without backgrounds

class DatasetCompositor:
    """Saves generated health bars directly and creates YOLO labels"""
    
    def __init__(self, backgrounds_dir: Path, output_dir: Path):
        self.output_dir = output_dir
        self.image_output = output_dir / "images"
        self.label_output = output_dir / "labels"
        self.backgrounds = []  # Not used in this version
        
        # Create output directories
        self.image_output.mkdir(parents=True, exist_ok=True)
        self.label_output.mkdir(parents=True, exist_ok=True)
    
    def compose_and_label(self, healthbar: np.ndarray, metadata: Dict, sample_id: int) -> bool:
        """
        Save raw health bar image and create YOLO label
        
        Args:
            healthbar: The generated health bar image (BGR with alpha)
            metadata: Dict with team, frame, hp_pct, hp_x, hp_y, hp_w, hp_h, width, height
            sample_id: Index of this sample
            
        Returns:
            True if saved successfully, False otherwise
        """
        try:
            # Extract metadata
            team = metadata.get("team", "unknown")
            frame = metadata.get("frame", "unknown")
            hp_x = metadata.get("hp_x", 0)
            hp_y = metadata.get("hp_y", 0)
            hp_w = metadata.get("hp_w", 100)
            hp_h = metadata.get("hp_h", 10)
            img_w = metadata.get("width", 150)
            img_h = metadata.get("height", 30)
            
            # Get YOLO class ID from team
            yolo_class = CLASS_TO_ID.get(frame, 0)  # Use frame config's yolo_class if available
            
            # If frame config exists, get the proper class ID
            if frame in VALID_FRAMES:
                yolo_class_name = VALID_FRAMES[frame].get("yolo_class", "Player Champion")
                yolo_class = CLASS_TO_ID.get(yolo_class_name, 0)
            
            # Save image
            img_filename = f"healthbar_{sample_id:05d}.png"
            img_path = self.image_output / img_filename
            
            # Remove alpha channel for JPEG/PNG saving
            if healthbar.shape[2] == 4:
                healthbar_bgr = healthbar[:, :, :3]
            else:
                healthbar_bgr = healthbar
            
            cv2.imwrite(str(img_path), healthbar_bgr)
            
            # Create YOLO label: class_id center_x center_y norm_w norm_h (all normalized 0-1)
            center_x = (hp_x + hp_w / 2) / img_w
            center_y = (hp_y + hp_h / 2) / img_h
            norm_w = hp_w / img_w
            norm_h = hp_h / img_h
            
            # Save label
            label_filename = f"{sample_id:05d}.txt"
            label_path = self.label_output / label_filename
            
            with open(label_path, "w") as f:
                f.write(f"{yolo_class} {center_x:.6f} {center_y:.6f} {norm_w:.6f} {norm_h:.6f}\n")
            
            return True
            
        except Exception as e:
            print(f"    Error saving sample {sample_id}: {e}")
            return False
    
    def _apply_augmentations(self, image: np.ndarray) -> np.ndarray:
        """Apply data augmentations: blur, compression, brightness"""
        result = image.copy()
        
        # 20% chance: slight blur (simulates stream compression)
        if random.random() < 0.20:
            kernel_size = random.choice([3, 5])
            result = cv2.GaussianBlur(result, (kernel_size, kernel_size), 0)
        
        # 15% chance: JPEG compression artifact simulation
        if random.random() < 0.15:
            # Encode/decode to simulate compression
            _, jpeg = cv2.imencode('.jpg', result, [cv2.IMWRITE_JPEG_QUALITY, random.randint(70, 90)])
            result = cv2.imdecode(jpeg, cv2.IMREAD_COLOR)
        
        # 10% chance: brightness adjustment
        if random.random() < 0.10:
            brightness = random.uniform(0.85, 1.15)
            result = cv2.convertScaleAbs(result, alpha=brightness, beta=0)
        
        return result

# Initialize compositor
compositor = DatasetCompositor(BACKGROUNDS_DIR, OUTPUT_DIR)
print("✓ Dataset Compositor initialized (saving raw health bars only)")

✓ Dataset Compositor initialized (saving raw health bars only)


In [1589]:
def generate_synthetic_dataset(num_samples: int = 1000, verbose: bool = True):
    """
    Main pipeline: Generate num_samples synthetic health bars
    
    Distribution:
    - 40% Player Champions (green)
    - 25% Enemy Champions (red)
    - 15% Ally Champions (blue)
    - 10% Turrets
    - 5% Epic Monsters
    - 5% Jungle Monsters
    """
    
    # Define distribution
    class_distribution = [
        ("player", 0.40),
        ("enemy", 0.25),
        ("ally", 0.15),
        ("player", 0.10),  # Turrets show as player bars
        ("enemy", 0.05),   # Epic monsters show as enemy bars
        ("ally", 0.05),    # Jungle monsters show as ally bars
    ]
    
    frame_choices = {
        "player": ["standard_champion", "yuumi_special"],
        "enemy": ["standard_champion", "turret", "epic_monster"],
        "ally": ["standard_champion", "jungle_monster_big", "structure"]
    }
    
    successful = 0
    failed = 0
    creation_fails = 0
    compose_fails = 0
    
    print(f"\n{'='*60}")
    print(f"GENERATING {num_samples} SYNTHETIC HEALTH BARS")
    print(f"{'='*60}")
    
    for i in range(num_samples):
        # Pick team and frame
        team = random.choices(
            [t for t, _ in class_distribution],
            weights=[w for _, w in class_distribution]
        )[0]
        
        frame_name = random.choice(frame_choices.get(team, list(VALID_FRAMES.keys())))
        
        # Generate healthbar
        healthbar, meta = factory.create_healthbar(
            hp_percentage=None,  # Random
            mana_percentage=None,  # Random
            team=team,
            frame_name=frame_name
        )
        
        if healthbar is None:
            creation_fails += 1
            failed += 1
            if i < 5:
                print(f"  Sample {i}: FAILED at healthbar creation (team={team}, frame={frame_name})")
            continue
        
        # Save health bar directly
        if compositor.compose_and_label(healthbar, meta, i):
            successful += 1
        else:
            compose_fails += 1
            failed += 1
            if i < 5:
                print(f"  Sample {i}: FAILED at saving")
        
        # Progress indicator
        if (i + 1) % max(1, num_samples // 20) == 0:
            pct = ((i + 1) / num_samples) * 100
            print(f"  [{pct:3.0f}%] Generated {i+1:5d} samples...")
    
    print(f"\n{'='*60}")
    print(f"GENERATION COMPLETE")
    print(f"  ✓ Successful:         {successful}")
    print(f"  ✗ Failed (creation):  {creation_fails}")
    print(f"  ✗ Failed (saving):    {compose_fails}")
    print(f"  → Output:    {OUTPUT_DIR}")
    print(f"    - Images:  {compositor.image_output}")
    print(f"    - Labels:  {compositor.label_output}")
    print(f"{'='*60}\n")
    
    return successful, failed

In [1590]:
# ==================== TEST THE THREE FIXES ====================

print("\n✓ Testing the three rendering fixes:\n")

# Test 1: Turret with gaps (plating)
print("1. Turret with plating gaps (NEW - transparent holes):")
turret_bar, meta = factory.create_healthbar(hp_percentage=50, mana_percentage=None, team="enemy", frame_name="turret")
if turret_bar is not None:
    print(f"   ✓ Turret generated: {meta['width']}x{meta['height']}")
    print(f"   ✓ Uses: Transparent plating gaps (alpha punch-through)")
else:
    print(f"   ✗ Turret generation failed")

# Test 2: Champion at 100% (edge-to-edge) with OLD tick marks
print("\n2. Champion at 100% HP (OLD - tick marks):")
champion_full, meta = factory.create_healthbar(hp_percentage=100, mana_percentage=100, team="player", frame_name="standard_champion")
if champion_full is not None:
    print(f"   ✓ Champion 100%: {meta['width']}x{meta['height']}")
    print(f"   ✓ Uses: Tick mark lines (every 100 HP)")
else:
    print(f"   ✗ Champion generation failed")

# Test 3: Yuumi with OLD tick marks
print("\n3. Yuumi special (OLD - tick marks):")
yuumi_bar, meta = factory.create_healthbar(hp_percentage=75, mana_percentage=50, team="player", frame_name="yuumi_special")
if yuumi_bar is not None:
    print(f"   ✓ Yuumi generated: {meta['width']}x{meta['height']}")
    print(f"   ✓ Uses: Tick mark lines (every 100 HP)")
else:
    print(f"   ✗ Yuumi generation failed")

# Test 4: Champion at various percentages (OLD tick marks)
print("\n4. Champion at various HP percentages (OLD - tick marks):")
for hp_pct in [5, 25, 50, 75, 100]:
    champion, meta = factory.create_healthbar(hp_percentage=hp_pct, mana_percentage=50, team="player", frame_name="standard_champion")
    if champion is not None:
        print(f"   ✓ {hp_pct:3d}%: {meta['width']}x{meta['height']}")

print("\n✓ All fixes applied correctly!")
print("   - Players & Yuumi: Tick mark lines (old way)")
print("   - Turrets: Transparent plating gaps (new way)\n")


✓ Testing the three rendering fixes:

1. Turret with plating gaps (NEW - transparent holes):
   ✓ Turret generated: 192x34
   ✓ Uses: Transparent plating gaps (alpha punch-through)

2. Champion at 100% HP (OLD - tick marks):
   ✓ Champion 100%: 137x29
   ✓ Uses: Tick mark lines (every 100 HP)

3. Yuumi special (OLD - tick marks):
   ✓ Yuumi generated: 141x34
   ✓ Uses: Tick mark lines (every 100 HP)

4. Champion at various HP percentages (OLD - tick marks):
   ✓   5%: 136x29
   ✓  25%: 136x29
   ✓  50%: 136x29
   ✓  75%: 136x29
   ✓ 100%: 136x29

✓ All fixes applied correctly!
   - Players & Yuumi: Tick mark lines (old way)
   - Turrets: Transparent plating gaps (new way)



In [1591]:
# Reinitialize compositor to save RAW health bars only (no backgrounds)
compositor = DatasetCompositor(BACKGROUNDS_DIR, OUTPUT_DIR)

# Clear previous failed attempts
import shutil
for f in IMAGES_DIR.glob("*.png"):
    f.unlink()
for f in LABELS_DIR.glob("*.txt"):
    f.unlink()
print("✓ Cleared previous outputs\n")

# Run generation again - NOW will save raw health bars
print(">>> Generating 100 raw health bar samples...\n")
success, fail = generate_synthetic_dataset(num_samples=100, verbose=True)

print(f"\n✓ Generated {success} health bar images!")
print(f"✓ Location: {IMAGES_DIR}")
print(f"\nYou can now manually inspect and adjust the health bars before adding backgrounds.\n")

✓ Cleared previous outputs

>>> Generating 100 raw health bar samples...


GENERATING 100 SYNTHETIC HEALTH BARS
  [  5%] Generated     5 samples...
  [ 10%] Generated    10 samples...
  [ 15%] Generated    15 samples...
  [ 20%] Generated    20 samples...
  [ 25%] Generated    25 samples...
  [ 30%] Generated    30 samples...
  [ 35%] Generated    35 samples...
  [ 40%] Generated    40 samples...
  [ 45%] Generated    45 samples...
  [ 50%] Generated    50 samples...
  [ 55%] Generated    55 samples...
  [ 60%] Generated    60 samples...
  [ 65%] Generated    65 samples...
  [ 70%] Generated    70 samples...
  [ 75%] Generated    75 samples...
  [ 80%] Generated    80 samples...
  [ 85%] Generated    85 samples...
  [ 90%] Generated    90 samples...
  [ 95%] Generated    95 samples...
  [100%] Generated   100 samples...

GENERATION COMPLETE
  ✓ Successful:         100
  ✗ Failed (creation):  0
  ✗ Failed (saving):    0
  → Output:    /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/synt

In [1592]:
def inspect_generated_dataset(num_samples: int = 5):
    """Preview generated samples"""
    image_files = sorted(list(compositor.image_output.glob("*.png")))[:num_samples]
    
    print(f"\n>>> Inspecting {len(image_files)} samples:\n")
    
    for img_file in image_files:
        # Extract sample ID from filename (healthbar_00000.png → 00000)
        sample_id = img_file.stem.replace("healthbar_", "")
        label_file = compositor.label_output / f"{sample_id}.txt"
        
        img = cv2.imread(str(img_file))
        if img is not None:
            h, w = img.shape[:2]
            
            if label_file.exists():
                with open(label_file, "r") as f:
                    label_data = f.read().strip().split()
                    class_id = int(label_data[0])
                    class_name = YOLO_CLASSES.get(class_id, "Unknown")
                    center_x, center_y, norm_w, norm_h = map(float, label_data[1:])
                    
                    # Convert to pixel coords
                    px = int(center_x * w)
                    py = int(center_y * h)
                    pw = int(norm_w * w)
                    ph = int(norm_h * h)
            else:
                class_id = -1
                class_name = "Unknown"
                px = py = pw = ph = 0
            
            print(f"  {img_file.name}")
            print(f"    Size: {w}x{h}")
            print(f"    Class: [{class_id}] {class_name}")
            print(f"    Box: center=({px}, {py}), size={pw}x{ph}")
            print()


def create_train_val_split(train_ratio: float = 0.8, val_ratio: float = 0.1):
    """
    Create train/val/test split directory structure
    Standard YOLO format:
    dataset/
      images/
        train/
        val/
        test/
      labels/
        train/
        val/
        test/
    """
    
    image_files = list(compositor.image_output.glob("*.png"))
    total = len(image_files)
    
    train_count = int(total * train_ratio)
    val_count = int(total * val_ratio)
    test_count = total - train_count - val_count
    
    splits = {
        "train": image_files[:train_count],
        "val": image_files[train_count:train_count + val_count],
        "test": image_files[train_count + val_count:]
    }
    
    print(f"\nCreating train/val/test split:")
    print(f"  Total samples: {total}")
    print(f"  Train (80%): {len(splits['train'])}")
    print(f"  Val (10%):   {len(splits['val'])}")
    print(f"  Test (10%):  {len(splits['test'])}")
    
    split_dir = OUTPUT_DIR / "split"
    
    for split_name, files in splits.items():
        # Create directories
        (split_dir / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (split_dir / "labels" / split_name).mkdir(parents=True, exist_ok=True)
        
        # Copy files
        for img_file in files:
            sample_id = img_file.stem.replace("healthbar_", "")
            label_file = compositor.label_output / f"{sample_id}.txt"
            
            import shutil
            shutil.copy(str(img_file), str(split_dir / "images" / split_name / img_file.name))
            if label_file.exists():
                shutil.copy(str(label_file), str(split_dir / "labels" / split_name / label_file.name))
    
    print(f"\n✓ Split dataset created at: {split_dir}")
    return split_dir


def create_dataset_yaml(split_dir: Path):
    """Create data.yaml for YOLOv8 training"""
    
    yaml_content = f"""path: {split_dir.absolute()}
train: images/train
val: images/val
test: images/test

nc: {len(YOLO_CLASSES)}
names: {list(YOLO_CLASSES.values())}
"""
    
    yaml_file = split_dir / "data.yaml"
    with open(yaml_file, "w") as f:
        f.write(yaml_content)
    
    print(f"\n✓ Created data.yaml at: {yaml_file}")
    return yaml_file


# Inspect first few samples
inspect_generated_dataset(num_samples=3)

# Create splits
split_dir = create_train_val_split()
yaml_file = create_dataset_yaml(split_dir)


>>> Inspecting 3 samples:

  healthbar_00000.png
    Size: 141x34
    Class: [0] Player Champion
    Box: center=(82, 16), size=104x13

  healthbar_00001.png
    Size: 192x34
    Class: [3] Turret
    Box: center=(95, 13), size=168x14

  healthbar_00002.png
    Size: 136x29
    Class: [0] Player Champion
    Box: center=(79, 11), size=107x13


Creating train/val/test split:
  Total samples: 100
  Train (80%): 80
  Val (10%):   10
  Test (10%):  10

✓ Split dataset created at: /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/synthetic_dataset/split

✓ Created data.yaml at: /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/synthetic_dataset/split/data.yaml


In [1593]:
# ==================== PRODUCTION DATASET GENERATION ====================
# Run this cell to generate a full training dataset

def generate_full_production_dataset(num_samples: int = 5000):
    """
    Generate production-scale dataset
    
    YOLO Recommendations for object detection:
    - Minimum: 1,000 images per class
    - Better: 5,000-10,000 images per class
    - Ideal: 20,000+ images per class
    
    With 5,000 total samples and 6 classes:
    - ~830 samples per class (minimum viable)
    - Increase to 10,000+ for better generalization
    """
    
    print("""
    ╔════════════════════════════════════════════════════════════════╗
    ║       LEAGUE OF LEGENDS HEALTH BAR DATASET GENERATOR           ║
    ║                                                                ║
    ║  Generating synthetic training data for YOLOv8 object         ║
    ║  detection model trained to recognize health bars             ║
    ║                                                                ║
    ║  6 Classes:                                                    ║
    ║    [0] Player Champion        [3] Turret/Structure            ║
    ║    [1] Ally Champion          [4] Epic Monster (Baron/Dragon) ║
    ║    [2] Enemy Champion         [5] Jungle Monster              ║
    ╚════════════════════════════════════════════════════════════════╝
    """)
    
    success, fail = generate_synthetic_dataset(num_samples=num_samples, verbose=True)
    
    print(f"""
    ╔════════════════════════════════════════════════════════════════╗
    ║  DATASET READY FOR YOLOV8 TRAINING                            ║
    ║                                                                ║
    ║  Output Structure:                                             ║
    ║  {OUTPUT_DIR}/                                          ║
    ║    images/  → All {success} generated samples                   ║
    ║    labels/  → Corresponding YOLO .txt labels                  ║
    ║    split/                                                      ║
    ║      images/ train/val/test/                                  ║
    ║      labels/ train/val/test/                                  ║
    ║    data.yaml → YOLOv8 configuration                           ║
    ║                                                                ║
    ║  Training with YOLOv8:                                         ║
    ║  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ ║
    ║  from ultralytics import YOLO                                 ║
    ║  model = YOLO('yolov8n.pt')                                   ║
    ║  results = model.train(                                       ║
    ║      data='{split_dir}/data.yaml',                            ║
    ║      epochs=100,                                              ║
    ║      imgsz=640,                                               ║
    ║      device=0  # GPU                                          ║
    ║  )                                                            ║
    ║                                                                ║
    ║                                                                ║
    ╚════════════════════════════════════════════════════════════════╝
    """)
    
    return success, fail

# Uncomment below to generate full production dataset (5000+ samples)
# This will take several minutes depending on CPU/GPU
# success, fail = generate_full_production_dataset(num_samples=5000)

In [1594]:
# ==================== DIAGNOSTICS ====================
# Debug why no images are being created

print("=" * 70)
print("DIAGNOSTIC REPORT")
print("=" * 70)

print(f"\n1. DIRECTORIES:")
print(f"   BASE_DIR:        {BASE_DIR} → {BASE_DIR.exists()}")
print(f"   COLORS_DIR:      {COLORS_DIR} → {COLORS_DIR.exists()}")
print(f"   FRAMES_DIR:      {FRAMES_DIR} → {FRAMES_DIR.exists()}")
print(f"   BACKGROUNDS_DIR: {BACKGROUNDS_DIR} → {BACKGROUNDS_DIR.exists()}")

print(f"\n2. VALID FRAMES LOADED:")
print(f"   Count: {len(VALID_FRAMES)}/{len(FRAME_CONFIGS)}")
for name, cfg in VALID_FRAMES.items():
    print(f"   ✓ {name}")

print(f"\n3. ASSET LIBRARY STATUS:")
print(f"   HP Color Teams: {list(assets.hp_colors.keys())}")
for team, color_list in assets.hp_colors.items():
    print(f"   - {team}: {len(color_list)} colors")
print(f"   Special Effects: {list(assets.special_effects.keys())}")

print(f"\n4. BACKGROUNDS LOADED:")
print(f"   Count: {len(compositor.backgrounds)}")
if BACKGROUNDS_DIR.exists():
    files = list(BACKGROUNDS_DIR.glob("*.png"))
    print(f"   Found PNG files: {len(files)}")
    for f in files[:5]:
        print(f"     - {f.name}")
    if len(files) > 5:
        print(f"     ... and {len(files) - 5} more")

print(f"\n5. GENERATION RESULTS:")
print(f"   Generated images: {success}")
print(f"   Failed attempts: {fail}")

print(f"\n6. OUTPUT STATUS:")
print(f"   Total files created:")
image_count = len(list(IMAGES_DIR.glob("*.png")))
label_count = len(list(LABELS_DIR.glob("*.txt")))
print(f"   - Images: {image_count}")
print(f"   - Labels: {label_count}")

print("\n" + "=" * 70)

DIAGNOSTIC REPORT

1. DIRECTORIES:
   BASE_DIR:        /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar → True
   COLORS_DIR:      /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/Classified/Colors → True
   FRAMES_DIR:      /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/Classified/Frames → True
   BACKGROUNDS_DIR: /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/crops → True

2. VALID FRAMES LOADED:
   Count: 9/9
   ✓ standard_champion
   ✓ execute_threshold
   ✓ invulnerable
   ✓ spellshield
   ✓ yuumi_special
   ✓ turret
   ✓ epic_monster
   ✓ jungle_monster_big
   ✓ structure

3. ASSET LIBRARY STATUS:
   HP Color Teams: ['player', 'ally', 'enemy']
   - player: 1 colors
   - ally: 1 colors
   - enemy: 1 colors
   Special Effects: ['shield', 'grey_health', 'spellshield', 'kled', 'energy', 'fury', 'hp_gradient']

4. BACKGROUNDS LOADED:
   Count: 0
   Found PNG files: 2
     - allyhp.png
     - enemyhp.png

5. GENERATION RESULTS:
   Generated images: 100
   Failed attempts: 0

6. OUTPUT 

In [1595]:
# ==================== EXHAUSTIVE BACKGROUND TEST SET GENERATOR ====================
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import random
import shutil

def generate_exhaustive_background_test_set():
    test_output_dir = OUTPUT_DIR / "test_combinations_backgrounds"
    test_images_dir = test_output_dir / "images"
    test_labels_dir = test_output_dir / "labels"
    
    # Purge old test images to keep the directory clean
    if test_images_dir.exists(): shutil.rmtree(test_images_dir)
    if test_labels_dir.exists(): shutil.rmtree(test_labels_dir)
    test_images_dir.mkdir(parents=True, exist_ok=True)
    test_labels_dir.mkdir(parents=True, exist_ok=True)

    # Fully expanded frame layouts mapping to all valid team color targets
    frame_test_configs = {
        "standard_champion": ["player", "ally", "enemy"],
        "execute_threshold": ["player", "enemy"],          # <-- ALLY REMOVED!
        "invulnerable": ["player", "ally", "enemy"],
        "spellshield": ["player", "enemy"],
        "yuumi_special": ["player", "ally", "enemy"],
        "turret": ["enemy"],
        "epic_monster": ["enemy"],
        "jungle_monster_big": ["ally"],
        "structure": ["enemy"]
    }   
    
    # Define allowed Kled frames
    valid_champ_frames = [
        "standard_champion", "execute_threshold", 
        "invulnerable", "spellshield", "yuumi_special"
    ]
    
    health_levels = {"low_hp": 15.0, "medium_hp": 50.0, "full_hp": 100.0}
    
    bg_files = list(BACKGROUNDS_DIR.glob("*.png")) + list(BACKGROUNDS_DIR.glob("*.jpg"))
    if not bg_files: return print("✗ Add background crops first.")

    print(">>> Running Matrix Verification Suite on new special asset classes...")
    test_records = []
    
    for bg_path in bg_files:
        bg_img_base = cv2.imread(str(bg_path), cv2.IMREAD_COLOR)
        if bg_img_base is None: continue
        
        for frame_name, teams in frame_test_configs.items():
            for team in teams:
                for hp_label, hp_pct in health_levels.items():
                    
                    # 1. Base status condition triggers
                    is_kled_test = False
                    kled_health = 0.0
                    
                    # <-- UPDATED: Removed the 'team == "enemy"' restriction so Kled can pair with Yuumi on player/ally
                    if frame_name in valid_champ_frames and random.random() > 0.5:
                        is_kled_test = True
                        kled_health = 25.0  
                        hp_pct = hp_pct * 0.6  
                        
                    normal_shield = 25.0 if hp_label == "medium_hp" else 0.0
                    spell_shield = 30.0 if (frame_name == "spellshield" or hp_label == "low_hp") else 0.0
                    grey_health = 20.0 if hp_label == "low_hp" else 0.0
                    
                    # Structure Overrides
                    if frame_name in ["turret", "structure"]:
                        spell_shield = 0.0
                        grey_health = 0.0
                    
                    # Run factory generation
                    healthbar, meta = factory.create_healthbar(
                        hp_percentage=hp_pct,
                        mana_percentage=60.0,
                        team=team,
                        frame_name=frame_name,
                        shield_pct=normal_shield,
                        spellshield_pct=spell_shield,
                        grey_hp_pct=grey_health,
                        kled_hp_pct=kled_health 
                    )
                    
                    if healthbar is None: continue
                    
                    # Canvas Compositing Layout Calculations
                    bg_composite = bg_img_base.copy()
                    bg_h, bg_w = bg_composite.shape[:2]
                    hb_h, hb_w = healthbar.shape[:2]
                    
                    if bg_h < hb_h + 40 or bg_w < hb_w + 40:
                        bg_composite = cv2.resize(bg_composite, (hb_w + 100, hb_h + 100))
                        bg_h, bg_w = bg_composite.shape[:2]
                        
                    paste_x = (bg_w - hb_w) // 2
                    paste_y = (bg_h - hb_h) // 2
                    
                    bg_roi = bg_composite[paste_y:paste_y+hb_h, paste_x:paste_x+hb_w]
                    alpha = healthbar[:, :, 3:4] / 255.0
                    bg_composite[paste_y:paste_y+hb_h, paste_x:paste_x+hb_w] = (
                        healthbar[:, :, :3] * alpha + bg_roi * (1.0 - alpha)
                    ).astype(np.uint8)
                    
                    # YOLO Labels
                    yolo_class_name = VALID_FRAMES[frame_name].get("yolo_class", "Player Champion")
                    yolo_class_id = CLASS_TO_ID.get(yolo_class_name, 0)
                    center_x = (paste_x + meta["hp_x"] + meta["hp_w"] / 2) / bg_w
                    center_y = (paste_y + meta["hp_y"] + meta["hp_h"] / 2) / bg_h
                    norm_w = meta["hp_w"] / bg_w
                    norm_h = meta["hp_h"] / bg_h
                    
                    # Unique explicit strings
                    file_id = f"verify_{bg_path.stem}_{frame_name}_{team}_{hp_label}_kled{is_kled_test}_ns{int(normal_shield)}_ss{int(spell_shield)}_gh{int(grey_health)}"
                    cv2.imwrite(str(test_images_dir / f"{file_id}.png"), bg_composite)
                    with open(test_labels_dir / f"{file_id}.txt", "w") as f:
                        f.write(f"{yolo_class_id} {center_x:.6f} {center_y:.6f} {norm_w:.6f} {norm_h:.6f}\n")
                    
                    # Log for the CSV manifest
                    test_records.append({
                        "filename": f"{file_id}.png",
                        "frame_style": frame_name,
                        "team_color": team,
                        "health_condition": hp_label,
                        "is_kled": is_kled_test,
                        "normal_shield": normal_shield,
                        "spell_shield": spell_shield,
                        "grey_health": grey_health,
                        "yolo_class_id": yolo_class_id
                    })
                        
    print(f"✓ New test frames written cleanly to: {test_images_dir}")
    
    # Save a structured verification manifest
    manifest_df = pd.DataFrame(test_records)
    csv_path = test_output_dir / "test_manifest_backgrounds.csv"
    manifest_df.to_csv(csv_path, index=False)
    
    return csv_path

# Execute testing automation
manifest_file = generate_exhaustive_background_test_set()

>>> Running Matrix Verification Suite on new special asset classes...
✓ New test frames written cleanly to: /Volumes/Kingston 2TB/proyectoLol/LolHealth-Bar/synthetic_dataset/test_combinations_backgrounds/images
